## 1. Imports & Load Raw Data

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os.path as path

In [121]:
df = pd.read_csv('dataset/amazon_delivery.csv')
y_target = 'Delivery_Time'
df.head(5)

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


## 2. Handle Missing / Invalid Values

In [122]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Order_Time'] = pd.to_timedelta(df['Order_Time'].astype(str), errors='coerce')
df['Pickup_Time'] = pd.to_timedelta(df['Pickup_Time'].astype(str), errors='coerce')

In [123]:
missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print(f'Missing Feature:\n{missing_feature}')

Missing values: 236
Duplicates: 0
Missing Feature:
['Agent_Rating', 'Order_Time', 'Weather']


In [124]:
for col in missing_feature:
    missing_percentage = (df[col].isnull().sum() / len(df)) * 100
    
    if missing_percentage > 5.0:
        df = df.drop(columns=col)
    else:
        if pd.api.types.is_timedelta64_dtype(df[col]):
            median_timedelta = df[col].median()
            df[col] = df[col].fillna(median_timedelta)
        
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            median_date = df[col].median()
            df[col] = df[col].fillna(median_date)

        elif df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            skewness = df[col].skew()
            if abs(skewness) < 0.5:
                df[col] = df[col].fillna(df[col].mean()).round(2)
            else:
                df[col] = df[col].fillna(df[col].median()).round(2)

missing_feature = df.columns[df.isnull().any()].to_list()
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing Feature:\n{missing_feature}')


Missing values: 0
Missing Feature:
[]


## 3. Handling Duplicated

In [125]:
df = df.drop_duplicates(keep='last')

jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah data duplikat: {jumlah_duplikat}")

Jumlah data duplikat: 0


## 4. Analisis & Handling Outliers

In [126]:
feature_numerik = df.select_dtypes(include=np.number).columns.drop(y_target)
Q1 = df[feature_numerik].quantile(0.25)
Q3 = df[feature_numerik].quantile(0.75)
IQR = Q3-Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sebelum Dihapus: {outliers.shape[0]}")

#delete outliers
df = df.loc[((df[feature_numerik] >= lower_bound) & (df[feature_numerik] <= upper_bound)).all(axis=1)]
outliers = df.loc[((df[feature_numerik] < lower_bound) | (df[feature_numerik] > upper_bound)).any(axis=1)]
print(f"Jumlah Outlier Sesudah Dihapus: {outliers.shape[0]}")

Jumlah Outlier Sebelum Dihapus: 8645
Jumlah Outlier Sesudah Dihapus: 0


## 5. Feature Engineering

### 5.1 Drop Feature ID

In [127]:
df = df.drop(columns=['Order_ID'])

### 5.2 Encoding Bins in Feature Numeric

In [128]:
batas_umur = [12, 25, 45, 60]
nama_label = ['Remaja', 'Dewasa', 'Lansia']
df['umur_category'] = pd.cut(df['Agent_Age'],bins=batas_umur,labels=nama_label,include_lowest=True)

labels = ['Needs Improvement', 'Average', 'Good', 'Top Performer']
df['Agent_Categori'] = pd.qcut(df['Agent_Rating'],q=4,labels=labels)

### 5.3 Create Feature Numeric

In [129]:
def haversine(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return c * 6371  # Jari-jari bumi dalam km

df['Distance_KM'] = haversine(df['Store_Latitude'], df['Store_Longitude'],df['Drop_Latitude'], df['Drop_Longitude']) 
df['Rating_per_KM'] = df['Agent_Rating'] / (df['Distance_KM'] + 0.1)
df['Age_Distance_Interaction'] = df['Agent_Age'] * df['Distance_KM']
df['Rating_to_Age_Ratio'] = df['Agent_Rating'] / df['Agent_Age']
df['Agent_Score'] = df['Agent_Rating'] * np.log1p(df['Rating_to_Age_Ratio'])

df = df.drop(columns=['Store_Latitude','Store_Longitude','Drop_Latitude','Drop_Longitude'])

### 5.4 Feature Datetime Engineering 

In [130]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
order_time_delta = pd.to_timedelta(df['Order_Time'].astype(str))
pickup_time_delta = pd.to_timedelta(df['Pickup_Time'].astype(str))

df['Order_Datetime'] = df['Order_Date'] + order_time_delta
df['Pickup_Datetime'] = df['Order_Date'] + pickup_time_delta

mask = df['Pickup_Datetime'] < df['Order_Datetime']
df.loc[mask, 'Pickup_Datetime'] += pd.Timedelta(days=1)

df['Prep_Time_Minutes'] = ((df['Pickup_Datetime'] - df['Order_Datetime']).dt.total_seconds() / 60.0) * df['Rating_to_Age_Ratio']
df['Order_Hour'] = df['Order_Datetime'].dt.hour
df['DayOfWeek'] = df['Order_Datetime'].dt.day_name()
df['Is_Weekend'] = np.where(df['Order_Datetime'].dt.dayofweek.isin([5, 6]), 'Weekend', 'Weekday')

df = df.drop(columns=['Order_Date', 'Order_Time', 'Pickup_Time','Order_Datetime', 'Pickup_Datetime'])

## 6.Save Cleaned Dataset

In [131]:
file_path = 'dataset/amazon_delivery_CLEANING.csv'

if not path.exists(file_path):
    df.to_csv(file_path, index=False)
    print('File belum ada. Berhasil menyimpan dataset baru!')
else:
    print('File sudah ada. Proses penyimpanan CSV dilewati (skip)')

df.head()

File belum ada. Berhasil menyimpan dataset baru!


,Agent_Age,Agent_Rating,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,umur_category,Agent_Categori,Distance_KM,Rating_per_KM,Age_Distance_Interaction,Rating_to_Age_Ratio,Agent_Score,Prep_Time_Minutes,Order_Hour,DayOfWeek,Is_Weekend
0,37,4.9,Sunny,High,motorcycle,Urban,120,Clothing,Dewasa,Good,3.025149,1.567925,111.930524,0.132432,0.609403,1.986486,11,Saturday,Weekend
1,34,4.5,Stormy,Jam,scooter,Metropolitian,165,Electronics,Dewasa,Needs Improvement,20.183530,0.221855,686.240011,0.132353,0.559340,0.661765,19,Friday,Weekday
2,23,4.4,Sandstorms,Low,motorcycle,Urban,130,Sports,Remaja,Needs Improvement,1.552758,2.662217,35.713429,0.191304,0.770215,2.869565,8,Saturday,Weekend
3,38,4.7,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,Dewasa,Average,7.790401,0.595660,296.035252,0.123684,0.548080,1.236842,18,Tuesday,Weekday
4,32,4.6,Cloudy,High,scooter,Metropolitian,150,Toys,Dewasa,Needs Improvement,6.210138,0.728986,198.724415,0.143750,0.617837,2.156250,13,Saturday,Weekend
